# DerivKit: ForecastKit — DALI Likelihood Demo

### Summary
This notebook demonstrates how to use **DerivKit's ForecastKit** to build local
DALI (Derivative Approximation of the Likelihood, see arXiv:1401.6892)
expansions of Gaussian log-likelihoods and compare them to:
- the **exact** log-likelihood,
- the **quadratic Fisher approximation**,
- the **doublet-DALI approximation**, and
- the **triplet-DALI approximation**.

We include both **1D** and **2D** DALI examples. The 1D example illustrates how
the Fisher, doublet-DALI, and triplet-DALI
approximations track an exact likelihood in a simple single-parameter case,
while the 2D example uses a nonlinear two-parameter toy model to generate
DALI samples and visualize the resulting contour structure.


### Usage
If you prefer to run the script version from the command line, you can use:

```bash
$ python -m scripts.forecast-kit-dali-1d --method adaptive
$ python -m scripts.forecast-kit-dali-1d --method adaptive --plot
```

In this notebook, just run the cells from top to bottom. You can change the
derivative backend (e.g. `"adaptive"`, `"finite"`) via the `method` variable
in the configuration cell below. Since each derivative backend supports its
own keyword arguments and tuning parameters, make sure to check the
documentation for the relevant options.

### What it does

- Defines a simple 1D model $o(x) = 100\,e^{x^2}$.
- Sets a fiducial parameter value $x_0$ and a Gaussian data covariance
  with scalar variance $\sigma_o^2$.
- Uses :class:`ForecastKit` to compute, at $x_0$:
  - the Fisher matrix $F$,
  - the doublet-DALI tensors $D_1$ and $D_2$,
  - the triplet-DALI tensors $T_1$, $T_2$, and $T_3$.
- Evaluates on 1D grids:
  - the **exact** Gaussian log-likelihood,
  - the **Fisher** quadratic approximation,
  - the **doublet-DALI** approximation including cubic and quartic terms,
  - the **triplet-DALI** approximation including higher-order corrections.
- Plots all curves on the same axes for visual comparison and prints a sanity
  check at $x = x_0$, where all approximations agree by construction.

### Notes

- In 1D, the tensors have shapes:
  - $F \to (1,1)$,
  - $D_1 \to (1,1,1)$,
  - $D_2 \to (1,1,1,1)$,
  - $T_1 \to (1,1,1,1)$,
  - $T_2 \to (1,1,1,1,1)$,
  - $T_3 \to (1,1,1,1,1,1)$.
- The code reads them as scalars `F[0,0]`, `D1[0,0,0]`, `D2[0,0,0,0]`,
  `T1[0,0,0,0]`, `T2[0,0,0,0,0]`, and `T3[0,0,0,0,0,0]`.
- When $x_0 \neq 0$, the odd-order DALI terms are generally non-zero, so the
  DALI approximation is typically asymmetric relative to the Fisher Gaussian
  and tends to track the exact likelihood better away from $x_0$.

### Requirements

- `derivkit` installed and importable in your Python environment.


In [ ]:
from getdist import plots as getdist_plots
import numpy as np
import matplotlib.pyplot as plt
from derivkit import ForecastKit

# Optional: if you have DerivKit's plotting style helpers available, you can use them.
try:
    from utils.style import DEFAULT_COLORS, apply_plot_style
except ImportError:  # fallback if running outside the repo
    DEFAULT_COLORS = {
        "blue": "#1f77b4",
        "red": "#d62728",
        "yellow": "#ff7f0e",
    }

    def apply_plot_style():
        plt.style.use("default")
        plt.rcParams.update(
            {
                "figure.figsize": (8, 5),
                "axes.grid": False,
                "font.size": 14,
            }
        )

print("Imports complete.")

In [ ]:
def test_model_1d(param_list) -> np.ndarray:
    """Returns one observable with a quadratic dependence on one parameter.

    Model:
        o(x) = 100 * exp(x^2)

    Args:
        param_list: list of one parameter [x].

    Returns:
        np.ndarray
            One-element array [o].
    """
    x = float(param_list[0])
    obs = 1e2 * np.exp(x**2)
    return np.array([obs], dtype=float)


def loglike_1d_exact(sigma_o: float, fiducial_x: float, x: float) -> float:
    """Exact Gaussian log-likelihood for one observable with variance sigma_o^2.

    log L(x) = -0.5 * ((o(x) - o(x0)) / sigma_o)^2

    Args:
        sigma_o:
            Standard deviation of the observable.
        fiducial_x:
            Fiducial parameter value x0.
        x:
            Parameter value at which to evaluate the log-likelihood.

    Returns:
        float:
            Exact log-likelihood value.
    """
    delta_o = test_model_1d([x]) - test_model_1d([fiducial_x])
    return float(-0.5 * (delta_o[0] / sigma_o) ** 2)


def loglike_1d_approx(tensors: list[np.ndarray], fiducial_x: float, x: float) -> float:
    """Approximate log-likelihood using Fisher + doublet-DALI terms.

    In 1D, the expansion reads:
        log L(x) ≈ -1/2 F Δx^2 - 1/2 G Δx^3 - 1/8 H Δx^4,
    where Δx = x - x0.

    Args:
        tensors:
            List of tensors [F, G, H] where
            F is (1,1), G is (1,1,1), H is (1,1,1,1) in 1D.
        fiducial_x:
            Fiducial parameter value x0.
        x:
            Parameter value at which to evaluate the log-likelihood.

    Returns:
        float:
            Approximate log-likelihood value.
    """
    dx = x - fiducial_x
    loglike = 0.0

    if len(tensors) >= 1:
        f = np.asarray(tensors[0])
        loglike += -0.5 * float(f[0, 0]) * dx**2

    if len(tensors) >= 3:
        g = np.asarray(tensors[1])
        h = np.asarray(tensors[2])
        loglike += -0.5 * float(g[0, 0, 0]) * dx**3
        loglike += -0.125 * float(h[0, 0, 0, 0]) * dx**4

    return float(loglike)


print("Model and likelihood helpers defined.")

In [ ]:
# Configuration

fiducial_values = [0.1]  # x0
covmat = np.array([[1.0]], dtype=float)  # C = [[sigma_o^2]] with sigma_o = 1

fiducial_x = float(fiducial_values[0])  # our assumed "true" parameter value
sigma_o = float(np.sqrt(covmat[0, 0]))  # standard deviation of the observable

# Derivative backend for ForecastKit ("adaptive", "finite", etc.)
method = "adaptive"

# Grids
xgrid = np.linspace(-1.0, 1.0, 1000)
xgrid_sparse = np.linspace(-0.2, 0.2, 100)

print(f"fiducial_x = {fiducial_x}")
print(f"sigma_o = {sigma_o}")
print(f"method = '{method}'")

In [ ]:
# Build ForecastKit object and compute tensors

observables = test_model_1d

fk = ForecastKit(
    function=observables,
    theta0=np.array(fiducial_values, dtype=float),
    cov=covmat,
)

# Fisher directly
fisher_matrix = fk.fisher(method=method)  # shape (1, 1)

# DALI tensors
dali = fk.dali(method=method, forecast_order=3)

# Extract tensors
dali_f = dali[1][0]  # Fisher matrix, shape (p, p)
dali_d1, dali_d2 = dali[2]  # shapes (p,p,p), (p,p,p,p)
dali_t1, dali_t2, dali_t3 = dali[3]  # triplet-DALI tensors

print("Fisher matrix from fk.fisher:\n", fisher_matrix)
print("\nFisher matrix from fk.dali:\n", dali_f)

print("\nDALI D1 tensor shape:", dali_d1.shape)
print("DALI D2 tensor shape:", dali_d2.shape)

print("\nDALI T1 tensor shape:", dali_t1.shape)
print("DALI T2 tensor shape:", dali_t2.shape)
print("DALI T3 tensor shape:", dali_t3.shape)

In [ ]:
# Evaluate exact and approximate log-likelihoods on grids

tensors = [fisher_matrix, dali_d1, dali_d2]

exact_like = np.array([loglike_1d_exact(sigma_o, fiducial_x, x) for x in xgrid])
fisher_like = np.array([loglike_1d_approx([fisher_matrix], fiducial_x, x) for x in xgrid])
dali_like_sparse = np.array([loglike_1d_approx(tensors, fiducial_x, x) for x in xgrid_sparse])

print("Computed exact, Fisher, and DALI log-likelihoods on the grids.")

In [ ]:
# Plot comparison: exact vs Fisher vs Doublet-DALI in 1D

apply_plot_style()
fig, ax = plt.subplots()

blue = DEFAULT_COLORS.get("blue", "#1f77b4")
red = DEFAULT_COLORS.get("red", "#d62728")
yellow = DEFAULT_COLORS.get("yellow", "#ff7f0e")

ax.plot(xgrid, exact_like, label="Exact Likelihood", linewidth=3, color=red)
ax.plot(xgrid, fisher_like, label="Fisher Matrix", linewidth=3, linestyle="-", color=yellow)
ax.plot(xgrid_sparse, dali_like_sparse, label="Doublet DALI",
        markersize=6, color=blue, linestyle="--", linewidth=3)

ax.set_title(r"$\mathrm{observable} = 100 \cdot e^{x^2}$", fontsize=20)
ax.set_xlabel(r"$x$", fontsize=20)
ax.set_ylabel(r"$\log P$", fontsize=20)
ax.set_xlim(-0.3, 0.6)
ax.set_ylim(-0.65, 0.05)
ax.legend(fontsize=14, framealpha=1.0)
ax.minorticks_off()

plt.tight_layout()
plt.show()

## 2D DALI vs Fisher contours (bonus)

As a bonus, this cell builds a simple **two-parameter** model and compares
the closed $1\sigma/2\sigma$contours of

- the **exact** Gaussian log-likelihood,
- the **Fisher quadratic** approximation, and
- the **Doublet-DALI** approximation,

using Matplotlib's `contour` on a common $(\theta_1, \theta_2)$ grid.

This is conceptually the same as the 1D demo above, but now in 2D so that
we can draw proper contour lines in parameter space.


In [ ]:
# 2D example: DALI contours from DerivKit samples
dk_red = "#f21901"
line_width = 1.5


def model_2d(theta):
    """Nonlinear 2D toy model with curved degeneracy."""
    x, eps = float(theta[0]), float(theta[1])

    k = 3.0
    a = 4.0
    c = 6.0

    o1 = 1e2 * np.exp((x - k * eps) ** 2) * np.exp(a * eps)
    o2 = 4e1 * np.exp(0.5 * x) * (1.0 + 0.3 * eps + c * eps**3)

    return np.array([o1, o2], dtype=float)


# Fiducial parameters and covariance
theta0 = np.array([0.18, 0.02], dtype=float)
cov = np.array(
    [
        [1.0, 0.95],
        [0.95, 1.0],
    ],
    dtype=float,
)

# Build ForecastKit and compute DALI tensors
fk = ForecastKit(function=model_2d, theta0=theta0, cov=cov)

dali = fk.dali(method=method, forecast_order=2)
F = dali[1][0]
D1, D2 = dali[2]

print("2D Fisher matrix F:\n", F)
print("\n2D DALI D1 tensor shape:", D1.shape)
print("2D DALI D2 tensor shape:", D2.shape)

# Draw DALI samples directly from DerivKit
samples = fk.getdist_dali_emcee(
    dali=dali,
    names=["x", "eps"],
    labels=[r"x", r"\epsilon"],
    label="DALI",
)

# Plot contours
plotter = getdist_plots.get_subplot_plotter(width_inch=5)
plotter.settings.linewidth_contour = line_width
plotter.settings.linewidth = line_width
plotter.settings.figure_legend_frame = False
plotter.settings.legend_rect_border = False
plotter.settings.axes_labelsize = 25

plotter.triangle_plot(
    [samples],
    params=["x", "eps"],
    filled=False,
    contour_colors=[dk_red],
    contour_lws=[line_width],
    contour_ls=["-"],
)

samples.numrows > 0